In [ ]:
import os
from datetime import datetime, timedelta

import git

REPOS = [
    "ClimaCoupler.jl",
    "ClimaAtmos.jl",
    "ClimaCore.jl",
    "ClimaTimesteppers.jl",
    "Thermodynamics.jl",
    "RRTMGP.jl",
]


def get_latest_commit_at_date(repo_path: str, date: str) -> dict | None:
    """Return the latest commit hash of the repository as of a specific date.

    If no commits were made, return None.

    Note that we include commits made on the given date by adding one day to
    the ``until`` parameter.
    """
    # Don't allow running for the current day, since that can cause
    # irreproducible results
    if datetime.fromisoformat(date).date() >= datetime.now().date():
        raise ValueError("Date must be in the past")
    repo = git.Repo(repo_path)
    repo.remotes.origin.fetch()
    commits = repo.iter_commits(
        "origin/main",
        until=datetime.fromisoformat(date) + timedelta(days=1),
    )
    for commit in commits:
        return {
            "rev": commit.hexsha,
            "timestamp": commit.committed_datetime.isoformat(),
        }


def get_repo_revs_at_date(date: str) -> dict:
    """Return a dictionary mapping repository names to their respective commit
    hashes at a specific date.
    """
    commits = {}
    for repo in REPOS:
        repo_path = os.path.join("../repos", repo)
        rev = get_latest_commit_at_date(repo_path, date)
        if rev is None:
            raise ValueError(
                f"No commits found for repository {repo} at date {date}"
            )
        commits[repo] = rev
    return commits


get_repo_revs_at_date("2026-01-25")